# Fase 1 - Preparación geométrica del corredor ferroviario

Objetivo:
- cargar la línea ferroviaria exportada desde QGIS,
- reproyectarla a un sistema métrico,
- fusionarla en un eje continuo,
- dividirla en bloques de 1 km,
- generar un buffer de 100 m por bloque,
- exportar las geometrías resultantes para las futuras peticiones ráster a Copernicus.

Entrada:
- `data/soria_torralba_axis.geojson`

Salidas:
- `outputs/centerline_blocks_wgs84.geojson`
- `outputs/corridor_blocks_wgs84.geojson`

In [1]:
# Imports, rutas y parámetros del preprocesado geométrico

import math
from pathlib import Path

import geopandas as gpd
from shapely.ops import unary_union, linemerge, substring

CORRIDOR_ID = "soria_torralba"

INPUT_GEOJSON = Path(f"../data/{CORRIDOR_ID}_axis.geojson")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

SEGMENT_LENGTH_M = 1000
BUFFER_M = 100
RAIL_FOOTPRINT_HALF_WIDTH_M = 2
RAIL_ALERT_HALF_WIDTH_M = 10

CRS_INPUT = "EPSG:4326"
CRS_METRIC = "EPSG:25830"
CRS_OUTPUT = "EPSG:4326"

In [2]:
# Carga del GeoJSON exportado desde QGIS, reproyección a CRS métrico y fusión de la línea

gdf = gpd.read_file(INPUT_GEOJSON)

if gdf.crs is None:
    gdf = gdf.set_crs(CRS_INPUT)

gdf_metric = gdf.to_crs(CRS_METRIC)

merged = unary_union(gdf_metric.geometry)
merged = linemerge(merged)

print("Corredor:", CORRIDOR_ID)
print("Número de features originales:", len(gdf))
print("CRS original:", gdf.crs)
print("Longitud total de la línea (m):", round(merged.length, 2))
print("Longitud total de la línea (km):", round(merged.length / 1000, 3))

Corredor: soria_torralba
Número de features originales: 50
CRS original: EPSG:4326
Longitud total de la línea (m): 93258.08
Longitud total de la línea (km): 93.258


In [3]:
# Segmentación de la línea en bloques de 1 km y generación de IDs tipo soria_torralba_km0_1


def km_label(value_m):
    return int(math.floor(value_m / 1000))


def build_segment_id(corridor_id, start_m, end_m):
    start_km = km_label(start_m)
    end_km = max(start_km + 1, int(math.ceil(end_m / 1000)))
    return f"{corridor_id}_km{start_km}_{end_km}"


def segment_line(line, corridor_id, segment_length):
    total_length = line.length
    rows = []
    start = 0.0
    segment_index = 1

    while start < total_length:
        end = min(start + segment_length, total_length)
        seg = substring(line, start, end)

        rows.append(
            {
                "corridor_id": corridor_id,
                "segment_index": segment_index,
                "segment_id": build_segment_id(corridor_id, start, end),
                "start_m": round(start, 3),
                "end_m": round(end, 3),
                "start_km": round(start / 1000, 3),
                "end_km": round(end / 1000, 3),
                "length_m": round(end - start, 3),
                "geometry": seg,
            }
        )

        start = end
        segment_index += 1

    return rows


segments = segment_line(merged, CORRIDOR_ID, SEGMENT_LENGTH_M)

segments_gdf = gpd.GeoDataFrame(segments, crs=CRS_METRIC)
corridor_gdf = segments_gdf.copy()
corridor_gdf["geometry"] = corridor_gdf.geometry.buffer(
    BUFFER_M, cap_style=2, join_style=2
)

print("Número de bloques generados:", len(segments_gdf))
print(
    "Longitud del último bloque (m):",
    round(float(segments_gdf.iloc[-1]["length_m"]), 2),
)

segments_gdf[["segment_index", "segment_id", "start_km", "end_km", "length_m"]].head(10)

Número de bloques generados: 94
Longitud del último bloque (m): 258.08


,segment_index,segment_id,start_km,end_km,length_m
0,1,soria_torralba_km0_1,0.0,1.0,1000.0
1,2,soria_torralba_km1_2,1.0,2.0,1000.0
2,3,soria_torralba_km2_3,2.0,3.0,1000.0
3,4,soria_torralba_km3_4,3.0,4.0,1000.0
4,5,soria_torralba_km4_5,4.0,5.0,1000.0
5,6,soria_torralba_km5_6,5.0,6.0,1000.0
6,7,soria_torralba_km6_7,6.0,7.0,1000.0
7,8,soria_torralba_km7_8,7.0,8.0,1000.0
8,9,soria_torralba_km8_9,8.0,9.0,1000.0
9,10,soria_torralba_km9_10,9.0,10.0,1000.0


In [4]:
# Exportación de los segmentos centrales y de los corredores bufferizados a GeoJSON en WGS84

segments_wgs84 = segments_gdf.to_crs(CRS_OUTPUT)
corridor_wgs84 = corridor_gdf.to_crs(CRS_OUTPUT)

segments_out = OUTPUT_DIR / f"{CORRIDOR_ID}_centerline_blocks_wgs84.geojson"
corridor_out = OUTPUT_DIR / f"{CORRIDOR_ID}_corridor_blocks_wgs84.geojson"

segments_wgs84.to_file(segments_out, driver="GeoJSON")
corridor_wgs84.to_file(corridor_out, driver="GeoJSON")

print("Exportación completada")
print("Segmentos centrales:", segments_out)
print("Corredores bufferizados:", corridor_out)
print("Corredor:", CORRIDOR_ID)
print("Longitud total de la línea (km):", round(merged.length / 1000, 3))
print("Número total de bloques:", len(segments_gdf))
print("Longitud objetivo por bloque (m):", SEGMENT_LENGTH_M)
print("Buffer lateral (m):", BUFFER_M)

Exportación completada
Segmentos centrales: ..\outputs\soria_torralba_centerline_blocks_wgs84.geojson
Corredores bufferizados: ..\outputs\soria_torralba_corridor_blocks_wgs84.geojson
Corredor: soria_torralba
Longitud total de la línea (km): 93.258
Número total de bloques: 94
Longitud objetivo por bloque (m): 1000
Buffer lateral (m): 100


In [5]:
# 5. Generación y exportación de la vía como footprint real en metros

rail_footprint_gdf = segments_gdf.copy()
rail_footprint_gdf["geometry"] = rail_footprint_gdf.geometry.buffer(
    RAIL_FOOTPRINT_HALF_WIDTH_M,
    cap_style=2,
    join_style=2
)

rail_footprint_wgs84 = rail_footprint_gdf.to_crs(CRS_OUTPUT)

rail_footprint_out = OUTPUT_DIR / f"{CORRIDOR_ID}_rail_footprint_wgs84.geojson"
rail_footprint_wgs84.to_file(rail_footprint_out, driver="GeoJSON")

print("Footprint ferroviario exportado")
print("Archivo:", rail_footprint_out)
print("Semiancho usado (m):", RAIL_FOOTPRINT_HALF_WIDTH_M)
print("Ancho total representado (m):", RAIL_FOOTPRINT_HALF_WIDTH_M * 2)

Footprint ferroviario exportado
Archivo: ..\outputs\soria_torralba_rail_footprint_wgs84.geojson
Semiancho usado (m): 2
Ancho total representado (m): 4


In [6]:
# 6. Generación y exportación de la banda de alerta junto a la vía

rail_alert_band_gdf = segments_gdf.copy()
rail_alert_band_gdf["geometry"] = rail_alert_band_gdf.geometry.buffer(
    RAIL_ALERT_HALF_WIDTH_M,
    cap_style=2,
    join_style=2
)

rail_alert_band_wgs84 = rail_alert_band_gdf.to_crs(CRS_OUTPUT)

rail_alert_band_out = OUTPUT_DIR / f"{CORRIDOR_ID}_rail_alert_band_wgs84.geojson"
rail_alert_band_wgs84.to_file(rail_alert_band_out, driver="GeoJSON")

print("Banda de alerta exportada")
print("Archivo:", rail_alert_band_out)
print("Semiancho usado (m):", RAIL_ALERT_HALF_WIDTH_M)
print("Ancho total representado (m):", RAIL_ALERT_HALF_WIDTH_M * 2)

Banda de alerta exportada
Archivo: ..\outputs\soria_torralba_rail_alert_band_wgs84.geojson
Semiancho usado (m): 10
Ancho total representado (m): 20
